# Demo — Contexto longo: o acervo inteiro no prompt

**Objetivo:** mostrar a primeira linha RAGless dos slides 18 e 19, **sem busca e sem índice**: nada de chunking, embedding ou banco vetorial. O acervo inteiro vai no prompt, com cache para não pagar a leitura de novo a cada chamada. O slide diz quando isso cabe: acervo pequeno e estável, que cabe na janela do modelo.

1. **O acervo cabe na janela?** → os 14 documentos num texto só, medidos em tokens;
2. **Perguntar sem busca** → três perguntas do gabarito, inclusive a multissalto, com o acervo inteiro no prompt;
3. **O cache em ação** → quanto da entrada de cada chamada veio do cache de prompt;
4. **Custo por pergunta x RAG** → o acervo inteiro contra 5 trechos: o custo migra para a consulta.

> **Cenário 100% fictício.** A Contoso Industrial e todos os documentos, projetos, pessoas e números do `acervo/` existem apenas para fins didáticos.

## 0. Preparação

Este notebook usa só a **Azure OpenAI v1 API** (cliente Python `OpenAI`). Não há índice, Azure AI Search nem embedding.

Variáveis necessárias:

- `AZURE_OPENAI_ENDPOINT`, no formato `https://<recurso>.openai.azure.com`
- `AZURE_OPENAI_API_KEY`
- `AZURE_OPENAI_DEPLOYMENT` (deployment de chat), de um modelo do GPT-4o até a família GPT-5.5, como o `gpt-5-mini` sugerido no `.env.example`. A Execução 3 mostra o cache de prompt desses modelos; da família GPT-5.6 em diante, as regras do cache mudam.

Os valores vêm do arquivo `Cap05/.env` (copie o `.env.example` e preencha); o que faltar é perguntado na hora da execução. As variáveis de embedding e do AI Search não são lidas aqui.

> Não grave chaves diretamente no notebook.

In [ ]:
%pip install -q "openai>=2" "python-dotenv>=1.0" "pandas>=2.1" "tiktoken>=0.7"

In [ ]:
import os
from getpass import getpass
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

load_dotenv(find_dotenv(usecwd=True))  # lê o .env da pasta Cap05, se existir

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT") or input(
    "AZURE_OPENAI_ENDPOINT (ex.: https://meu-recurso.openai.azure.com): "
).strip()
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY") or getpass("AZURE_OPENAI_API_KEY: ")
AZURE_OPENAI_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT") or input(
    "AZURE_OPENAI_DEPLOYMENT (deployment de chat): "
).strip()

client = OpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    base_url=f"{AZURE_OPENAI_ENDPOINT.rstrip('/')}/openai/v1/",
)

# Pasta Cap05: a primeira pasta, subindo a partir daqui, que contém o acervo.
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "acervo").is_dir())
PASTA_ACERVO = RAIZ / "acervo"

print("Cliente configurado.")
print("Deployment de chat:", AZURE_OPENAI_DEPLOYMENT)
print("Acervo:", PASTA_ACERVO)

### Funções auxiliares

- `carregar_trechos`: o mesmo chunking dos outros notebooks, um trecho por seção `## `. Aqui ele não busca nada: traduz o gabarito, que aponta seções, nos arquivos que a resposta deveria citar, e mede, na Execução 4, o contexto de um RAG.
- `PERGUNTAS` e `arquivos_esperados`: o gabarito (`acervo/perguntas.json`) e os arquivos de onde vêm os trechos que respondem cada pergunta.

In [ ]:
import re


def carregar_trechos(pasta_acervo: Path) -> list[dict]:
    """Lê os .md do acervo e devolve um trecho por seção "## " (chunking pela estrutura do documento)."""
    trechos = []
    for caminho in sorted(pasta_acervo.rglob("*.md")):
        fonte = caminho.relative_to(pasta_acervo).as_posix()
        doc_id = fonte.removesuffix(".md").replace("/", "_")
        texto = caminho.read_text(encoding="utf-8")
        titulo = texto.splitlines()[0].lstrip("# ").strip()
        # O que vem antes do primeiro "## " é o cabeçalho do documento (título e metadados).
        for numero, secao in enumerate(re.split(r"(?m)^## ", texto)[1:], start=1):
            nome_secao, _, conteudo = secao.partition("\n")
            trechos.append({
                "id": f"{doc_id}_{numero:02d}",
                "doc_id": doc_id,
                "titulo": titulo,
                "secao": nome_secao.strip(),
                "conteudo": conteudo.strip(),
                "fonte": fonte,
            })
    return trechos


trechos = carregar_trechos(PASTA_ACERVO)
print(f"{len(trechos)} trechos de {len({t['doc_id'] for t in trechos})} documentos")

In [ ]:
import json

PERGUNTAS = {p["id"]: p for p in json.loads((PASTA_ACERVO / "perguntas.json").read_text(encoding="utf-8"))}

# O gabarito aponta trechos (seções); com o acervo inteiro no prompt, a resposta só consegue citar o arquivo.
FONTE_DO_TRECHO = {t["id"]: t["fonte"] for t in trechos}


def arquivos_esperados(pid: str) -> list[str]:
    """Arquivos de origem dos trechos que o gabarito espera para a pergunta."""
    return sorted({FONTE_DO_TRECHO[i] for grupo in PERGUNTAS[pid]["trechos_esperados"] for i in grupo})

---

# Execução 1 — O acervo cabe na janela?

Sem busca, a primeira pergunta de arquitetura é de tamanho: o acervo inteiro precisa caber na **janela de contexto** do modelo, com folga para a pergunta e para a resposta.

Juntamos os 14 documentos num texto só, sempre na mesma ordem, com um cabeçalho `### arquivo: <caminho>` antes de cada um (é ele que permite à resposta citar o arquivo), e medimos tudo com o tokenizador `o200k_base`.

`JANELA_DO_MODELO` é uma referência: 128 mil tokens de entrada é a janela do `gpt-4o`; o `gpt-5-mini`, por exemplo, aceita 272 mil tokens de entrada (400 mil de contexto). Ajuste ao modelo do seu deployment.

In [ ]:
import tiktoken

pd.set_option("display.max_colwidth", None)  # tabelas mostram o texto inteiro

# Régua de tamanho: o tokenizador o200k_base, dos modelos GPT mais recentes.
codificador = tiktoken.get_encoding("o200k_base")
JANELA_DO_MODELO = 128_000  # ajuste ao seu modelo

# Sempre a mesma ordem de arquivos: é o que mantém o texto idêntico de uma chamada para outra.
arquivos = {
    caminho.relative_to(PASTA_ACERVO).as_posix(): caminho.read_text(encoding="utf-8").strip()
    for caminho in sorted(PASTA_ACERVO.rglob("*.md"))
}
ACERVO = "\n\n".join(f"### arquivo: {fonte}\n\n{texto}" for fonte, texto in arquivos.items())

display(pd.DataFrame([{"arquivo": fonte, "tokens": len(codificador.encode(texto))} for fonte, texto in arquivos.items()]))

tokens_acervo = len(codificador.encode(ACERVO))
ocupacao = f"{100 * tokens_acervo / JANELA_DO_MODELO:.1f}".replace(".", ",")
print(f"Acervo inteiro, com os cabeçalhos: {tokens_acervo} tokens ({len(arquivos)} arquivos)")
print(f"Janela do modelo: {JANELA_DO_MODELO} tokens. O acervo ocupa {ocupacao}% dela e caberia "
      f"{JANELA_DO_MODELO // tokens_acervo} vezes nela, sem contar a pergunta e a resposta.")

print("\nO começo do texto que vai no prompt:\n")
print(ACERVO[:330], "[...]")

### O que observar

- O acervo inteiro tem ~5 mil tokens: cabe com folga na janela, com espaço de sobra para a pergunta e a resposta.
- Só a norma de segurança da informação é quase um terço do acervo. Num acervo real, um único contrato ou manual pode ter dezenas de milhares de tokens.
- O cabeçalho `### arquivo:` é o único rastro de origem que o modelo recebe: a citação, aqui, só pode ser por arquivo.

### Leitura arquitetural

- Caber na janela é a primeira condição do slide 18 ("acervo pequeno e estável, que cabe na janela do modelo"), mas não a única: caber não quer dizer que compensa pagar a leitura de tudo a cada pergunta (Execução 4).
- Este acervo tem 14 documentos. Um acervo corporativo tem milhares, e aí nenhuma janela resolve: é caso de indexar.

---

# Execução 2 — Perguntar sem busca

Cada pergunta vai ao modelo com o acervo inteiro, e a ordem do prompt é a decisão que importa:

1. as **instruções** fixas: responder só com base no acervo e citar o arquivo;
2. o **acervo inteiro**, sempre o mesmo texto, na mesma ordem;
3. a **pergunta**, no fim.

Tudo o que vem antes da pergunta é idêntico em todas as chamadas: é esse **prefixo estável** que o cache de prompt reaproveita (Execução 3). Se a pergunta viesse antes do acervo, o trecho idêntico entre as chamadas seria só o das instruções, bem abaixo do mínimo de 1.024 tokens do cache.

`perguntar` devolve a resposta e os números de cada chamada: tokens de entrada, quantos deles vieram do cache, tokens de saída e latência. Fazemos três perguntas do gabarito, em sequência: o código de norma, a string exata e a multissalto.

### O que esperar e por que pode variar

- As três tendem a sair certas, citando o arquivo: não existe etapa de recuperação para errar. O modelo lê a NR-12 e a NR-10 lado a lado, e os SKUs 48213-B, C e D por inteiro.
- A multissalto também tende a sair. Os quatro elos estão no prompt: a suspensão da Rolamentos Alfa, o fornecedor exclusivo do SKU 48213-B, a troca preventiva na prensa P-400 e a dependência do projeto ATLAS. No RAG, é a pergunta que pede o grafo (degrau 4 do slide 17, notebook `tipos-de-rag/05-grafo`).
- O texto muda a cada execução, e o modelo pode ligar só parte da cadeia ou deixar de citar algum arquivo. Por isso a linha **Citados na resposta** conta os arquivos que o gabarito espera.

In [ ]:
import time

INSTRUCOES = """
Responda em português, exclusivamente com base no acervo fornecido.
Cite o arquivo de cada afirmação no formato [arquivo: caminho/do/arquivo.md].
Se o acervo não trouxer a informação, responda:
'Não há evidência suficiente no acervo.'
""".strip()


def perguntar(pergunta: str) -> dict:
    """Uma chamada sem busca: instruções e acervo primeiro (o prefixo estável), a pergunta no fim."""
    inicio = time.perf_counter()
    resposta = client.responses.create(
        model=AZURE_OPENAI_DEPLOYMENT,
        instructions=INSTRUCOES,
        input=f"ACERVO:\n{ACERVO}\n\nPERGUNTA:\n{pergunta}",
        store=False,
    )
    latencia = time.perf_counter() - inicio
    uso = resposta.usage
    detalhes = uso.input_tokens_details  # se o serviço não mandar o detalhamento, conta como 0
    return {
        "texto": resposta.output_text,
        "input_tokens": uso.input_tokens,
        "cached_tokens": (detalhes.cached_tokens if detalhes else 0) or 0,
        "output_tokens": uso.output_tokens,
        "latencia_s": round(latencia, 1),
    }


chamadas = []  # os números de cada chamada, para a tabela da Execução 3

for pid in ["codigo-de-norma", "string-exata", "multissalto"]:
    pergunta = PERGUNTAS[pid]["pergunta"]
    resultado = perguntar(pergunta)
    chamadas.append({"pergunta": pid, **resultado})

    # O modelo às vezes omite a pasta ou a extensão: basta o nome do arquivo aparecer na resposta.
    esperados = arquivos_esperados(pid)
    citados = [f for f in esperados if Path(f).stem in resultado["texto"]]
    lista = ", ".join(f"`{f}`" for f in esperados)

    display(Markdown(f"### {pergunta}"))
    display(Markdown(f"**Arquivos que o gabarito espera:** {lista}  \n**Citados na resposta:** {len(citados)} de {len(esperados)}"))
    display(Markdown(resultado["texto"]))

### O que observar

- Na multissalto, confira a linha **Citados na resposta**: ela mostra se o modelo ligou os quatro documentos da cadeia.
- Compare com o notebook `tipos-de-rag/01-vetorial-puro`: lá, o SKU 48213-B dependia da sorte do vetor. Aqui o texto exato está no prompt, ao lado dos SKUs vizinhos.
- A citação é por **arquivo**, não por seção: é tudo o que o cabeçalho `### arquivo:` permite.

### Leitura arquitetural

- Sem busca, não existe erro de recuperação. O roteiro de diagnóstico do slide 29 encurta, porque o trecho certo sempre está no prompt.
- O preço da simplicidade aparece em outro lugar: a rastreabilidade cai para o nível do arquivo, e **todo usuário recebe o acervo inteiro**. Não há onde aplicar o filtro por grupo do slide 26; respeitar permissão exigiria um prompt, e um cache, por perfil de acesso.

### Fala sugerida

> “Aqui o sistema não tem como trazer a norma errada: ele não escolhe nada, lê tudo. E é exatamente por isso que cada pergunta sai cara.”

---

# Execução 3 — O cache em ação

O Azure OpenAI guarda por alguns minutos o processamento do começo do prompt e o reaproveita quando a chamada seguinte começa igual. As regras do cache de prompt nos modelos deste notebook, do GPT-4o até a família GPT-5.5, segundo a documentação:

- é **automático**: não há parâmetro para ligar;
- vale para prompts de **1.024 tokens ou mais**, e os primeiros 1.024 precisam ser **idênticos**: um caractere diferente neles já zera o cache;
- o acerto aparece em `usage.input_tokens_details.cached_tokens` e cresce em blocos de 128 tokens depois dos primeiros 1.024;
- o cache em memória costuma ser apagado depois de 5 a 10 minutos sem uso, e sempre em até uma hora após o último uso. Ele não é compartilhado entre assinaturas do Azure;
- os tokens lidos do cache são cobrados **com desconto** sobre o preço de entrada, e o desconto varia conforme o modelo e o tipo de deployment.

Aqui o prefixo estável (instruções + acervo) tem uns 5 mil tokens, bem acima do mínimo. A tabela junta as três chamadas da Execução 2 e repete a primeira pergunta, agora com o prefixo já gravado.

### O que esperar e por que pode variar

- A **1ª chamada** tende a vir com 0 tokens em cache: é ela que grava o prefixo. Se o notebook rodou há poucos minutos, até ela pode vir do cache. No `gpt-5.5`, que já vem com a retenção estendida ligada, o mesmo pode acontecer depois de um ensaio feito horas antes: o cache pode durar até 24 horas.
- Da **2ª em diante**, a maior parte da entrada tende a vir do cache. Raramente chega a 100%: a pergunta, no fim, muda de uma chamada para outra, e o arredondamento em blocos de 128 deixa uma sobra.
- O acerto **não é garantido**: a própria documentação fala em melhorar a taxa de acerto, não em garanti-la. Se tudo vier zerado, rode a célula de novo, o que tende a mostrar o acerto; se continuar zerado, o modelo pode não ter cache de prompt.
- Se só a pergunta repetida acertar, o deployment provavelmente é de um modelo da família GPT-5.6 em diante, em que o cache só é lido nos pontos de corte (*breakpoints*). O padrão é o fim da última mensagem, que aqui já inclui a pergunta: perguntas diferentes não reaproveitam o acervo.
- Não espere uma queda grande de latência: com 5 mil tokens, o tempo de gerar a resposta (e de raciocinar, nos modelos GPT-5) tende a pesar mais que o de ler o prompt.

In [ ]:
# A mesma pergunta da 1ª chamada, agora com o prefixo já gravado no cache.
chamadas.append({"pergunta": "codigo-de-norma (repetida)", **perguntar(PERGUNTAS["codigo-de-norma"]["pergunta"])})

display(pd.DataFrame([
    {
        "chamada": i,
        "pergunta": c["pergunta"],
        "tokens de entrada": c["input_tokens"],
        "em cache": c["cached_tokens"],
        "% em cache": round(100 * c["cached_tokens"] / c["input_tokens"], 1),
        "tokens de saída": c["output_tokens"],
        "latência (s)": c["latencia_s"],
    }
    for i, c in enumerate(chamadas, start=1)
]))

### O que observar

- Compare a coluna **em cache** da 1ª chamada com a das seguintes: a 1ª pagou a leitura inteira e gravou o prefixo; as seguintes tendem a reler quase tudo do cache.
- A coluna **tokens de entrada** quase não muda: com ou sem cache, toda chamada manda o acervo inteiro. O cache muda o preço e a latência da leitura, não a quantidade lida.

### Leitura arquitetural

- O cache só funciona porque o prefixo é estável: instruções fixas, acervo na mesma ordem, pergunta no fim. Mudou um documento? O prefixo idêntico vai só até o ponto da mudança, e, se ela cair nos primeiros 1.024 tokens, o cache zera. É por isso que o slide pede acervo **estável**.
- O cache em memória é temporário: some depois de alguns minutos sem uso. Com poucas perguntas espalhadas ao longo do dia, quase toda chamada paga a leitura cheia. Alguns modelos oferecem retenção estendida, de até 24 horas. No `gpt-5.5`, ela já vem ligada por padrão: um ensaio feito horas antes pode deixar o cache ativo, e a 1ª chamada da aula pode já vir do cache.
- Nos modelos da família GPT-5.6 em diante, sem um ponto de corte depois do acervo, cada pergunta nova grava um cache próprio, e essa gravação pode ser cobrada. Reaproveitar o acervo entre perguntas diferentes exigiria um `prompt_cache_breakpoint` logo depois dele, com a entrada em blocos de conteúdo. Esse formato de entrada funciona também nos modelos anteriores; o erro 400 vem dos parâmetros `prompt_cache_breakpoint` e `prompt_cache_options`, que esses modelos não aceitam. Por isso, o ponto de corte fica fora deste exemplo. O `prompt_cache_key` sozinho não cria esse ponto.

---

# Execução 4 — Custo por pergunta: contexto longo x RAG

O contexto longo não gasta nada para indexar: não há chunking, embedding nem índice. Em compensação, **cada pergunta lê o acervo inteiro**. O RAG faz o contrário: paga a indexação uma vez e, na consulta, manda ao modelo só alguns trechos.

Comparamos os tokens de entrada de uma pergunta nos dois desenhos, com a mesma régua (`o200k_base`), as mesmas instruções e a mesma pergunta. Muda só o contexto: o acervo inteiro ou 5 trechos típicos, os 5 de tamanho mais próximo da mediana, montados como no `responder` dos notebooks de `tipos-de-rag`. Os trechos servem só para medir o tamanho: num RAG de verdade, seriam os 5 recuperados para a pergunta.

São tokens, não reais: o preço do token varia por modelo, região e tipo de deployment. A projeção mensal usa um volume de exemplo: `PERGUNTAS_POR_DIA` × 30 dias.

In [ ]:
PERGUNTAS_POR_DIA = 1_000  # volume de exemplo

# Os 5 trechos de tamanho mais próximo da mediana, no formato do responder() dos tipos-de-rag.
por_tamanho = sorted(trechos, key=lambda t: len(codificador.encode(t["conteudo"])))
meio = len(por_tamanho) // 2
contexto_rag = "\n\n".join(
    f"[{t['id']}] {t['titulo']} — {t['secao']}\n{t['conteudo']}" for t in por_tamanho[meio - 2:meio + 3]
)

pergunta = PERGUNTAS["string-exata"]["pergunta"]
entradas = {
    "Contexto longo: o acervo inteiro": f"ACERVO:\n{ACERVO}\n\nPERGUNTA:\n{pergunta}",
    "RAG: 5 trechos típicos": f"TRECHOS RECUPERADOS:\n{contexto_rag}\n\nPERGUNTA:\n{pergunta}",
}
tokens_por_pergunta = {
    desenho: len(codificador.encode(INSTRUCOES)) + len(codificador.encode(texto)) for desenho, texto in entradas.items()
}

display(pd.DataFrame([
    {
        "desenho": desenho,
        "tokens de entrada por pergunta": tokens,
        "milhões de tokens de entrada por mês": round(tokens * PERGUNTAS_POR_DIA * 30 / 1_000_000, 1),
    }
    for desenho, tokens in tokens_por_pergunta.items()
]))

longo, rag = tokens_por_pergunta.values()
print(f"O contexto longo lê {longo / rag:.0f} vezes mais tokens de entrada por pergunta que o RAG de 5 trechos.")

# Conferência com o que a API mediu nas Execuções 2 e 3.
total_entrada = sum(c["input_tokens"] for c in chamadas)
total_cache = sum(c["cached_tokens"] for c in chamadas)
print(f"Medido pela API: {total_entrada / len(chamadas):.0f} tokens de entrada por pergunta, "
      f"{100 * total_cache / total_entrada:.0f}% deles lidos do cache.")

### O que observar

- O fator depende do tamanho dos trechos. Os deste acervo são pequenos (~60 tokens de conteúdo); com trechos de 512 tokens, o ponto de partida do slide 11, 5 trechos somariam uns 2.600 tokens e o fator cairia para ~2.
- O que não depende do trecho: o contexto do RAG fica fixo, em k trechos, enquanto o acervo cresce; o do contexto longo cresce junto com o acervo. Dobrou o acervo, dobrou o custo de cada pergunta.
- A linha "Medido pela API" fecha a conta: o cache baixa o preço de parte desses tokens, mas eles são lidos em toda pergunta.

### Leitura arquitetural

- **O custo migra para a consulta.** O RAG paga na indexação (chunking, embedding, índice e o serviço de busca ligado o mês inteiro) e economiza em cada pergunta. O contexto longo não tem custo de indexação e paga em toda pergunta.
- Tokens de contexto por pergunta "é a métrica que a fatura enxerga" (slide 27). O slide 7 avisa que, no RAG, cada trecho recuperado é token de entrada, em toda chamada, para sempre; no contexto longo, o token de entrada de toda chamada é o acervo inteiro.
- Cabe com acervo pequeno, estável e sem permissão por usuário. Acervo grande estoura a janela e a fatura; acervo volátil quebra o cache; permissão por usuário exige um prompt por perfil. Nos três casos, o índice volta a valer.

### Fala sugerida

> “Cada variante troca flexibilidade por custo ou previsibilidade. Acervo grande, volátil ou com permissão por usuário pede o RAG clássico.”

---

# Quadro final — o que cada execução provou?

| Execução | O que vimos | Decisão arquitetural |
|---|---|---|
| 1 | O acervo inteiro tem ~5 mil tokens e cabe com folga na janela | Medir o acervo em tokens antes de escolher: caber na janela é a primeira condição |
| 2 | Sem busca não há erro de recuperação: até a multissalto tende a sair, com citação só por arquivo | Aceitar o que vem junto: citação por arquivo e o acervo inteiro para todo usuário |
| 3 | A partir da 2ª chamada, a maior parte da entrada tende a vir do cache | Prefixo estável no começo, pergunta no fim e acervo que muda pouco |
| 4 | Cada pergunta lê ~9 vezes mais tokens que um RAG de 5 trechos | Acervo grande, volátil ou com permissão por usuário: volta o índice |

## Frase de fechamento

> **O custo migra para a consulta. Com acervo grande, volátil ou com permissão por usuário, o índice volta a valer.**

---

# Referências oficiais

- Azure OpenAI — cache de prompt (prompt caching):  
  https://learn.microsoft.com/azure/foundry/openai/how-to/prompt-caching

- Azure OpenAI — Responses API:  
  https://learn.microsoft.com/azure/foundry/openai/how-to/responses

- Modelos do Azure OpenAI no Foundry — janela de contexto de cada modelo:  
  https://learn.microsoft.com/azure/foundry/foundry-models/concepts/models-sold-directly-by-azure

- Azure OpenAI — preços, inclusive o da entrada em cache:  
  https://azure.microsoft.com/pricing/details/azure-openai/

> Consulte o preço vigente do modelo, da região e do tipo de deployment antes da aula.